# Validação do Ensemble com Gate Conservador e Seleção Estrutural Ranqueada

Este notebook avalia em cinco trajetórias um ensemble com especialista local, validação preditiva conservadora e seleção estrutural ranqueada. O ganho preditivo é penalizado pelo erro-padrão entre cortes temporais. Para a decisão binária, o ranking-base ocupa no máximo 45% dos pares possíveis; pares adicionais só são resgatados quando têm bootstrap ≥ 0,60, rank preditivo ≥ 0,50 e suporte ≥ 0,75. A regra reduz estruturas excessivamente densas sem alterar `edge_probability`.

A métrica principal é **Average Precision (AP)**. F1, precision, recall, SHD, AUROC, taxa de vitórias e tempo são resultados secundários. O ensemble é comparado com cada algoritmo que o compõe. Com cinco trajetórias, a análise continua exploratória: o menor p-valor bilateral possível do Wilcoxon é `0,0625`.

> Limite da conclusão: cinco trajetórias podem sugerir vantagem e estimar custo, mas não demonstram superioridade estatística no nível de 5%. As trajetórias também compartilham o mesmo grafo, portanto não sustentam generalização universal.

## 1. Protocolo exploratório e critério confirmatório futuro

A configuração abaixo foi obtida durante desenvolvimento iterativo no Traffic e, portanto, não é pré-registrada. O critério confirmatório é mantido documentado para uma execução futura em grafos não usados no desenvolvimento. O ensemble seria considerado superior a cada algoritmo avulso na métrica principal somente se, simultaneamente:

1. o limite inferior do IC 95% do ganho médio de AP for maior que `0,02`;
2. o p-valor pareado de Wilcoxon, corrigido por Holm, for menor que `0,05`;
3. o ensemble vencer em pelo menos 70% das trajetórias pareadas.

A configuração usa `PCMCI + DYNOTEARS + NeuralGrangercMLP + VARLiNGAM`, oito bootstraps em blocos, limiar binário `0,50`, peso local `0,60`, penalização de redundância `0,20` e gate conservador com penalização `0,50`, Ridge `5,0` e expoente `0,35`. Nenhum desses cálculos consulta o grafo verdadeiro durante a execução.

In [1]:
from pathlib import Path
import importlib
import json
import os
import pickle
import time

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from causal_discovery import (
    CausalPreprocessor,
    build_complete_undirected_pair_scores,
    compute_paired_superiority_statistics,
    compute_ranked_undirected_skeleton_metrics,
    compute_undirected_skeleton_metrics,
    get_registered_method_kwargs,
    get_registered_method_weights,
    get_registered_methods,
    load_time_series_dataset,
)
import causal_discovery.benchmark as benchmark_module
import causal_discovery.ensemble_selection as ensemble_selection_module

# Recarrega os módulos quando o framework foi alterado no mesmo kernel Jupyter.
benchmark_module = importlib.reload(benchmark_module)
ensemble_selection_module = importlib.reload(ensemble_selection_module)
build_complete_undirected_pair_scores = benchmark_module.build_complete_undirected_pair_scores
compute_paired_superiority_statistics = benchmark_module.compute_paired_superiority_statistics
compute_ranked_undirected_skeleton_metrics = benchmark_module.compute_ranked_undirected_skeleton_metrics
compute_undirected_skeleton_metrics = benchmark_module.compute_undirected_skeleton_metrics
select_robust_ensemble_combination = (
    ensemble_selection_module.select_robust_ensemble_combination
)
add_ranked_structure_selection = (
    ensemble_selection_module.add_ranked_structure_selection
)

DATA_PATH = Path("datasets/causaltime/traffic/gen_data.npy")
GRAPH_PATH = Path("datasets/causaltime/traffic/graph.npy")
RESULTS_DIR = Path(".local/results/traffic_validation_ranked_rescue_10nodes_seed42_5traj")
BASE_CACHE_DIR = Path(".local/results/traffic_conditional_gate_cache")

N_NODES = 10
NODE_SAMPLE_SEED = 42
N_TRAJECTORIES = 5
TRAJECTORY_SAMPLE_SEED = 2026
CALIBRATION_TRAJECTORIES = [42, 207, 210, 312, 369]
ENSEMBLE_METHOD_NAMES = [
    "PCMCI", "DYNOTEARS", "NeuralGrangercMLP", "VARLiNGAM",
]
COMPARATOR_METHOD_NAMES = []
MAX_LAG = 2
ENSEMBLE_THRESHOLD = 0.50
N_BOOTSTRAP = 8
LOCAL_EXPERT_WEIGHT = 0.60
PREDICTIVE_VALIDATION_WEIGHT = 0.75
PREDICTIVE_VALIDATION_SPLITS = 3
PREDICTIVE_VALIDATION_RIDGE_ALPHA = 5.0
PREDICTIVE_VALIDATION_CONDITIONAL_PARENTS = 0
PREDICTIVE_VALIDATION_UNCERTAINTY_PENALTY = 0.50
PREDICTIVE_VALIDATION_RANK_EXPONENT = 0.35
RANKED_SELECTION_MAX_PAIR_DENSITY = 0.45
RANKED_SELECTION_RESCUE_BOOTSTRAP_MIN = 0.60
RANKED_SELECTION_RESCUE_PREDICTIVE_RANK_MIN = 0.50
RANKED_SELECTION_RESCUE_SUPPORT_MIN = 0.75
METHOD_REDUNDANCY_PENALTY = 0.20
RANDOM_STATE = 42
MEASURE_INDIVIDUAL_RUNTIMES = False

MINIMUM_AP_GAIN = 0.02
MINIMUM_WIN_RATE = 0.70
SIGNIFICANCE_LEVEL = 0.05
MIN_CONFIRMATORY_TRAJECTORIES = 10
STATISTICAL_BOOTSTRAPS = 10_000

OVERWRITE_CHECKPOINT = False
RETRY_FAILURES = True
STOP_ON_ERROR = False


## 2. Seleção de nós e amostra de trajetórias

São mantidos os mesmos dez nós sorteados por seed. Cinco trajetórias de calibração são removidas do universo de amostragem e outras cinco são sorteadas com seed `2026`. Como o desenvolvimento iterativo também inspecionou o desempenho do bloco final para rejeitar a versão condicional, estes resultados devem ser tratados como exploratórios, não como validação confirmatória independente. O `graph.npy` não entra no cálculo do ensemble, mas é usado depois para comparar as alternativas durante o desenvolvimento.

In [2]:
available_bundle = load_time_series_dataset(
    DATA_PATH,
    data_format="causaltime",
    graph_path=GRAPH_PATH,
    selected_columns=None,
    trajectory_index=0,
    column_prefix="traffic",
)
node_rng = np.random.default_rng(NODE_SAMPLE_SEED)
selected_nodes = sorted(node_rng.choice(
    available_bundle.available_columns,
    size=min(N_NODES, len(available_bundle.available_columns)),
    replace=False,
).tolist())

dataset_bundle = load_time_series_dataset(
    DATA_PATH,
    data_format="causaltime",
    graph_path=GRAPH_PATH,
    selected_columns=selected_nodes,
    trajectory_index=0,
    column_prefix="traffic",
)
nodes = list(dataset_bundle.selected_columns)
ground_truth = dataset_bundle.ground_truth.copy()

available_trajectory_indices = np.setdiff1d(
    np.arange(dataset_bundle.trajectory_count),
    np.asarray(CALIBRATION_TRAJECTORIES, dtype=int),
)
trajectory_rng = np.random.default_rng(TRAJECTORY_SAMPLE_SEED)
trajectory_indices = sorted(trajectory_rng.choice(
    available_trajectory_indices,
    size=min(N_TRAJECTORIES, dataset_bundle.trajectory_count),
    replace=False,
).tolist())

truth_summary = compute_undirected_skeleton_metrics(
    pd.DataFrame(columns=["source", "target", "lag"]),
    ground_truth,
    nodes=nodes,
)
print(f"Nós: {len(nodes)}")
print(f"Nós selecionados por seed {NODE_SAMPLE_SEED}: {nodes}")
print(f"Features temporais máximas no GES: {len(nodes) * (MAX_LAG + 1)}")
print(f"Pares possíveis: {truth_summary['candidate_pairs']}")
print(f"Pares verdadeiros: {truth_summary['ground_truth_pairs']}")
print(f"Prevalência: {truth_summary['ground_truth_prevalence']:.2%}")
print(f"Trajetórias de calibração excluídas: {CALIBRATION_TRAJECTORIES}")
print(f"Trajetórias selecionadas ({len(trajectory_indices)}): {trajectory_indices}")


Nós: 10
Nós selecionados por seed 42: ['traffic_00', 'traffic_01', 'traffic_03', 'traffic_06', 'traffic_08', 'traffic_09', 'traffic_12', 'traffic_13', 'traffic_14', 'traffic_19']
Features temporais máximas no GES: 30
Pares possíveis: 45
Pares verdadeiros: 8
Prevalência: 17.78%
Trajetórias de calibração excluídas: [42, 207, 210, 312, 369]
Trajetórias selecionadas (5): [12, 85, 174, 306, 406]


## 3. Fun??es do experimento

Para o ranking, o ensemble usa `ensemble_score`; métodos com p-valor usam `1 - p_value` e os demais usam o módulo do score. O gate preditivo compara, em cortes temporais expansivos, um modelo autorregressivo do alvo com o mesmo modelo acrescido da fonte. O ganho conservador é `média - 0,50 × erro-padrão`; ganhos instáveis perdem força. A decisão binária usa `ensemble_selected`: os pares são ordenados por `ensemble_score`, recebem teto-base de 45% e podem ser resgatados apenas por concordância conjunta de estabilidade, previsão e suporte. O gate e a seleção não consultam o `graph.npy` e preservam `edge_probability`.

In [3]:
def preprocess_trajectory(trajectory_index):
    raw = dataset_bundle.trajectory_frame(int(trajectory_index))
    preprocessor = CausalPreprocessor(
        raw, significance_level=0.05, decomposition_period=None
    )
    return preprocessor.fit_transform(
        make_stationary=True,
        normalize=True,
        remove_trend=False,
        max_diffs=2,
    )


def restrict_method_relations(method, allowed_relations):
    allowed_relations = set(allowed_relations)

    def run_restricted(data, **kwargs):
        result = method(data, **kwargs)
        if result is None or result.empty:
            return result
        mask = [
            (source, target) in allowed_relations
            for source, target in zip(result["source"], result["target"])
        ]
        return result.loc[mask].reset_index(drop=True)

    return run_restricted


def selection_arguments(processed_data):
    return {
        "min_methods": 4,
        "max_methods": 4,
        "min_votes": 1,
        "n_bootstrap": N_BOOTSTRAP,
        "block_size": max(2, len(processed_data) // 12),
        "stability_threshold": 0.60,
        "selection_probability_threshold": 0.50,
        "prior_edge_probability": 0.10,
        "posterior_weight": 0.70,
        "adaptive_method_weights": True,
        "stability_weight": 0.65,
        "local_expert_weight": LOCAL_EXPERT_WEIGHT,
        "predictive_validation_weight": PREDICTIVE_VALIDATION_WEIGHT,
        "predictive_validation_max_lag": MAX_LAG,
        "predictive_validation_splits": PREDICTIVE_VALIDATION_SPLITS,
        "predictive_validation_ridge_alpha": PREDICTIVE_VALIDATION_RIDGE_ALPHA,
        "predictive_validation_conditional_parents": PREDICTIVE_VALIDATION_CONDITIONAL_PARENTS,
        "predictive_validation_uncertainty_penalty": PREDICTIVE_VALIDATION_UNCERTAINTY_PENALTY,
        "predictive_validation_rank_exponent": PREDICTIVE_VALIDATION_RANK_EXPONENT,
        "ranked_selection_max_pair_density": RANKED_SELECTION_MAX_PAIR_DENSITY,
        "ranked_selection_rescue_bootstrap_min": RANKED_SELECTION_RESCUE_BOOTSTRAP_MIN,
        "ranked_selection_rescue_predictive_rank_min": RANKED_SELECTION_RESCUE_PREDICTIVE_RANK_MIN,
        "ranked_selection_rescue_support_min": RANKED_SELECTION_RESCUE_SUPPORT_MIN,
        "method_redundancy_penalty": METHOD_REDUNDANCY_PENALTY,
        "method_stability_power": 1.0,
        "method_diversity_bonus": 0.15,
        "method_density_penalty": 0.50,
        "minimum_method_weight": 0.05,
        "confidence_level": 0.95,
        "random_state": RANDOM_STATE,
        "precompute_runs": True,
        "parallel_jobs": max(1, min(4, (os.cpu_count() or 2) - 1)),
        "max_bootstrap_seconds": 900,
    }


def evidence_mode(frame):
    if "p_value" in frame.columns:
        p_values = pd.to_numeric(frame["p_value"], errors="coerce")
        if np.isfinite(p_values).any():
            return "one_minus_p_value"
    return "absolute_score"


def evaluate_strategy(frame, strategy, trajectory_index, runtime_seconds, *, probability=False):
    binary_frame = frame
    binary_threshold = ENSEMBLE_THRESHOLD
    if probability and "ensemble_selected" in frame:
        binary_frame = frame.loc[
            frame["ensemble_selected"].fillna(False).astype(bool)
        ].copy()
        binary_threshold = 0.0
    binary = compute_undirected_skeleton_metrics(
        binary_frame,
        ground_truth,
        prob_threshold=binary_threshold,
        nodes=nodes,
    )
    pair_scores = build_complete_undirected_pair_scores(
        frame,
        nodes,
        evidence=(
            "ensemble_score"
            if probability and "ensemble_score" in frame.columns
            else "probability" if probability
            else evidence_mode(frame)
        ),
    )
    ranked = compute_ranked_undirected_skeleton_metrics(pair_scores, ground_truth)
    return {
        "trajectory_index": int(trajectory_index),
        "strategy": str(strategy),
        "precision": binary["precision"],
        "recall": binary["recall"],
        "f1_score": binary["f1_score"],
        "structural_hamming_distance": binary["structural_hamming_distance"],
        "true_positives": binary["true_positives"],
        "false_positives": binary["false_positives"],
        "false_negatives": binary["false_negatives"],
        "average_precision": ranked["average_precision"],
        "roc_auc": ranked["roc_auc"],
        "runtime_seconds": float(runtime_seconds),
    }


def baseline_rows(trajectory_index):
    pairs = [(nodes[i], nodes[j]) for i in range(len(nodes)) for j in range(i + 1, len(nodes))]
    all_pairs = pd.DataFrame([
        {"source": source, "target": target, "lag": 1, "score": 1.0, "p_value": np.nan}
        for source, target in pairs
    ])
    rng = np.random.default_rng(RANDOM_STATE + int(trajectory_index))
    random_scores = rng.random(len(pairs))
    true_pair_count = truth_summary["ground_truth_pairs"]
    selected = np.argsort(random_scores)[-true_pair_count:]
    random_edges = pd.DataFrame([
        {"source": pairs[index][0], "target": pairs[index][1], "lag": 1,
         "score": random_scores[index], "p_value": np.nan}
        for index in selected
    ])
    random_pair_scores = pd.DataFrame([
        {"source": source, "target": target, "score": score}
        for (source, target), score in zip(pairs, random_scores)
    ])

    all_metrics = evaluate_strategy(all_pairs, "ALL_PAIRS", trajectory_index, 0.0)
    random_binary = compute_undirected_skeleton_metrics(random_edges, ground_truth, nodes=nodes)
    random_ranked = compute_ranked_undirected_skeleton_metrics(random_pair_scores, ground_truth)
    random_metrics = {
        "trajectory_index": int(trajectory_index), "strategy": "RANDOM_DENSITY",
        "precision": random_binary["precision"], "recall": random_binary["recall"],
        "f1_score": random_binary["f1_score"],
        "structural_hamming_distance": random_binary["structural_hamming_distance"],
        "true_positives": random_binary["true_positives"],
        "false_positives": random_binary["false_positives"],
        "false_negatives": random_binary["false_negatives"],
        "average_precision": random_ranked["average_precision"],
        "roc_auc": random_ranked["roc_auc"], "runtime_seconds": 0.0,
    }
    return [all_metrics, random_metrics]


def base_cache_signature(trajectory_index, processed_data):
    return {
        "trajectory_index": int(trajectory_index),
        "nodes": list(nodes),
        "processed_rows": len(processed_data),
        "max_lag": MAX_LAG,
        "n_bootstrap": N_BOOTSTRAP,
        "ensemble_methods": list(ENSEMBLE_METHOD_NAMES),
        "local_expert_weight": LOCAL_EXPERT_WEIGHT,
        "method_redundancy_penalty": METHOD_REDUNDANCY_PENALTY,
        "random_state": RANDOM_STATE,
    }


def run_trajectory(
    trajectory_index, methods, comparator_methods, method_kwargs, method_weights
):
    processed = preprocess_trajectory(trajectory_index)
    relations = {(source, target) for source in nodes for target in nodes if source != target}
    restricted = {
        name: restrict_method_relations(method, relations)
        for name, method in methods.items()
    }
    restricted_comparators = {
        name: restrict_method_relations(method, relations)
        for name, method in comparator_methods.items()
    }

    individual_outputs = {}
    individual_runtimes = {
        name: np.nan for name in [*restricted, *restricted_comparators]
    }
    if MEASURE_INDIVIDUAL_RUNTIMES:
        for name, method in restricted.items():
            started = time.perf_counter()
            individual_outputs[name] = method(processed, **method_kwargs[name])
            individual_runtimes[name] = time.perf_counter() - started

    BASE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = BASE_CACHE_DIR / f"trajectory_{int(trajectory_index):04d}.pkl"
    expected_cache_signature = base_cache_signature(trajectory_index, processed)
    cached_payload = None
    if cache_path.exists():
        with cache_path.open("rb") as stream:
            candidate_payload = pickle.load(stream)
        if candidate_payload.get("signature") == expected_cache_signature:
            cached_payload = candidate_payload

    started = time.perf_counter()
    selection = select_robust_ensemble_combination(
        processed,
        restricted,
        method_kwargs=method_kwargs,
        method_weights=method_weights,
        expert_knowledge=[],
        precomputed_outputs=(
            cached_payload.get("outputs") if cached_payload else None
        ),
        precomputed_bootstrap_outputs=(
            cached_payload.get("bootstrap_outputs") if cached_payload else None
        ),
        **selection_arguments(processed),
    )
    recalculation_runtime = time.perf_counter() - started
    cached_base_runtime = (
        cached_payload.get("base_runtime_seconds") if cached_payload else None
    )
    ensemble_runtime = (
        float(cached_base_runtime)
        if cached_base_runtime is not None and np.isfinite(cached_base_runtime)
        else recalculation_runtime
    )
    if cached_payload is None:
        with cache_path.open("wb") as stream:
            pickle.dump({
                "signature": expected_cache_signature,
                "outputs": selection["precomputed_outputs"],
                "bootstrap_outputs": selection["precomputed_bootstrap_outputs"],
                "base_runtime_seconds": recalculation_runtime,
            }, stream, protocol=pickle.HIGHEST_PROTOCOL)
    if not individual_outputs:
        for evaluation in selection["all_evaluations"].values():
            for name, output in evaluation["outputs"].items():
                individual_outputs.setdefault(name, output)
    for name, method in restricted_comparators.items():
        comparator_started = time.perf_counter()
        individual_outputs[name] = method(processed, **method_kwargs[name])
        individual_runtimes[name] = time.perf_counter() - comparator_started

    expected_outputs = set(restricted) | set(restricted_comparators)
    missing_outputs = sorted(expected_outputs - set(individual_outputs))
    if missing_outputs:
        raise RuntimeError(f"Saídas-base ausentes para: {missing_outputs}")

    rows = [
        evaluate_strategy(
            individual_outputs[name], name, trajectory_index, individual_runtimes[name]
        )
        for name in [*restricted, *restricted_comparators]
    ]
    summary = selection["best_evaluation"]["probabilistic_summary"].copy()
    summary_without_gate = summary.copy()
    summary_without_gate["ensemble_score"] = summary_without_gate[
        "pre_validation_ensemble_score"
    ]
    summary_without_gate = add_ranked_structure_selection(
        summary_without_gate, nodes=nodes,
        max_pair_density=RANKED_SELECTION_MAX_PAIR_DENSITY,
    )
    rows.append(evaluate_strategy(
        summary_without_gate, "ENSEMBLE_SEM_GATE", trajectory_index,
        ensemble_runtime, probability=True
    ))
    rows.append(evaluate_strategy(
        summary, "ENSEMBLE", trajectory_index, ensemble_runtime, probability=True
    ))
    rows.extend(baseline_rows(trajectory_index))
    weight_diagnostics = selection["best_evaluation"][
        "method_weight_diagnostics"
    ].set_index("method")
    selected_pair_count = len({
        tuple(sorted((str(row.source), str(row.target))))
        for row in summary.loc[summary["ensemble_selected"]].itertuples()
    })
    selection_row = {
        "trajectory_index": int(trajectory_index),
        "best_combination": " + ".join(selection["best_combination"]),
        "ensemble_runtime_seconds": ensemble_runtime,
        "cache_reused": cached_payload is not None,
        "recalculation_runtime_seconds": recalculation_runtime,
        "processed_rows": len(processed),
        "selected_pair_count": selected_pair_count,
        "ranked_selection_max_pair_density": RANKED_SELECTION_MAX_PAIR_DENSITY,
        "ranked_selection_rescue_bootstrap_min": RANKED_SELECTION_RESCUE_BOOTSTRAP_MIN,
        "ranked_selection_rescue_predictive_rank_min": RANKED_SELECTION_RESCUE_PREDICTIVE_RANK_MIN,
        "ranked_selection_rescue_support_min": RANKED_SELECTION_RESCUE_SUPPORT_MIN,
        "adaptive_weights": json.dumps(
            selection["best_evaluation"]["effective_method_weights"],
            ensure_ascii=False, sort_keys=True,
        ),
        "method_redundancy": json.dumps(
            weight_diagnostics["redundancy"].dropna().to_dict(),
            ensure_ascii=False, sort_keys=True,
        ),
        "dominant_method_counts": json.dumps(
            summary["dominant_method"].dropna().value_counts().to_dict(),
            ensure_ascii=False, sort_keys=True,
        ),
        "predictive_supported_edges": int((summary["predictive_gain"] > 0).sum()),
        "mean_predictive_gain": float(summary["predictive_gain"].mean()),
        "mean_predictive_gain_raw": float(summary["predictive_gain_mean"].mean()),
        "mean_predictive_standard_error": float(
            summary["predictive_gain_standard_error"].mean()
        ),
        "mean_predictive_positive_split_ratio": float(
            summary["predictive_positive_split_ratio"].mean()
        ),
    }
    return rows, selection_row


## 4. Execução com checkpoint

A validação usa cinco trajetórias e uma única combinação de quatro métodos. Os resultados são salvos em `.local/results/traffic_validation_ranked_rescue_10nodes_seed42_5traj`. As saídas caras dos algoritmos e bootstraps são cacheadas separadamente e podem ser reutilizadas para recalcular o gate e a seleção; o cache possui assinatura da configuração-base.

In [4]:
all_registered_methods = get_registered_methods()
all_method_kwargs = get_registered_method_kwargs(MAX_LAG)
all_method_weights = get_registered_method_weights()
methods = {name: all_registered_methods[name] for name in ENSEMBLE_METHOD_NAMES}
comparator_methods = {
    name: all_registered_methods[name] for name in COMPARATOR_METHOD_NAMES
}
method_kwargs = {
    name: all_method_kwargs[name]
    for name in [*ENSEMBLE_METHOD_NAMES, *COMPARATOR_METHOD_NAMES]
}
method_weights = {name: all_method_weights[name] for name in ENSEMBLE_METHOD_NAMES}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = RESULTS_DIR / "metrics.csv"
selections_path = RESULTS_DIR / "selections.csv"
failures_path = RESULTS_DIR / "failures.csv"
metadata_path = RESULTS_DIR / "configuration.json"

configuration = {
    "data_path": str(DATA_PATH), "graph_path": str(GRAPH_PATH),
    "nodes": nodes, "node_sample_seed": NODE_SAMPLE_SEED,
    "trajectory_indices": trajectory_indices,
    "max_lag": MAX_LAG, "n_bootstrap": N_BOOTSTRAP,
    "ensemble_threshold": ENSEMBLE_THRESHOLD,
    "local_expert_weight": LOCAL_EXPERT_WEIGHT,
    "predictive_validation_weight": PREDICTIVE_VALIDATION_WEIGHT,
    "predictive_validation_splits": PREDICTIVE_VALIDATION_SPLITS,
    "predictive_validation_ridge_alpha": PREDICTIVE_VALIDATION_RIDGE_ALPHA,
    "predictive_validation_conditional_parents": PREDICTIVE_VALIDATION_CONDITIONAL_PARENTS,
    "predictive_validation_uncertainty_penalty": PREDICTIVE_VALIDATION_UNCERTAINTY_PENALTY,
    "predictive_validation_rank_exponent": PREDICTIVE_VALIDATION_RANK_EXPONENT,
    "ranked_selection_max_pair_density": RANKED_SELECTION_MAX_PAIR_DENSITY,
    "ranked_selection_rescue_bootstrap_min": RANKED_SELECTION_RESCUE_BOOTSTRAP_MIN,
    "ranked_selection_rescue_predictive_rank_min": RANKED_SELECTION_RESCUE_PREDICTIVE_RANK_MIN,
    "ranked_selection_rescue_support_min": RANKED_SELECTION_RESCUE_SUPPORT_MIN,
    "method_redundancy_penalty": METHOD_REDUNDANCY_PENALTY,
    "ensemble_methods": list(methods),
    "comparator_methods": list(comparator_methods),
    "random_state": RANDOM_STATE,
    "measure_individual_runtimes": MEASURE_INDIVIDUAL_RUNTIMES,
    "minimum_ap_gain": MINIMUM_AP_GAIN,
    "minimum_win_rate": MINIMUM_WIN_RATE,
    "significance_level": SIGNIFICANCE_LEVEL,
    "min_confirmatory_trajectories": MIN_CONFIRMATORY_TRAJECTORIES,
}
if metadata_path.exists() and not OVERWRITE_CHECKPOINT:
    previous_configuration = json.loads(metadata_path.read_text(encoding="utf-8"))
    if previous_configuration != configuration:
        raise RuntimeError(
            "O checkpoint pertence a outra configuração. Altere RESULTS_DIR "
            "ou use OVERWRITE_CHECKPOINT=True conscientemente."
        )
if OVERWRITE_CHECKPOINT:
    for path in [metrics_path, selections_path, failures_path, metadata_path]:
        if path.exists():
            path.unlink()
metadata_path.write_text(
    json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8"
)

metrics_results = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
selection_results = pd.read_csv(selections_path) if selections_path.exists() else pd.DataFrame()
failure_results = pd.read_csv(failures_path) if failures_path.exists() else pd.DataFrame()
completed = set(
    metrics_results.loc[metrics_results.get("strategy", pd.Series(dtype=str)).eq("ENSEMBLE"),
                        "trajectory_index"].astype(int)
) if not metrics_results.empty else set()
failed = set(failure_results["trajectory_index"].astype(int)) if not failure_results.empty else set()

for position, trajectory_index in enumerate(trajectory_indices, start=1):
    if trajectory_index in completed or (trajectory_index in failed and not RETRY_FAILURES):
        print(f"[{position}/{len(trajectory_indices)}] trajetória {trajectory_index}: checkpoint")
        continue
    print(f"[{position}/{len(trajectory_indices)}] trajetória {trajectory_index}: executando...")
    started = time.perf_counter()
    try:
        rows, selection_row = run_trajectory(
            trajectory_index, methods, comparator_methods, method_kwargs, method_weights
        )
        metrics_results = pd.concat([metrics_results, pd.DataFrame(rows)], ignore_index=True)
        selection_results = pd.concat(
            [selection_results, pd.DataFrame([selection_row])], ignore_index=True
        )
        metrics_results.to_csv(metrics_path, index=False)
        selection_results.to_csv(selections_path, index=False)
        if not failure_results.empty:
            failure_results = failure_results.loc[
                failure_results["trajectory_index"].astype(int).ne(int(trajectory_index))
            ].reset_index(drop=True)
            if failure_results.empty:
                failures_path.unlink(missing_ok=True)
            else:
                failure_results.to_csv(failures_path, index=False)
        print(f"  concluída em {(time.perf_counter() - started) / 60:.1f} min")
    except Exception as error:
        failure_row = {
            "trajectory_index": int(trajectory_index),
            "error_type": type(error).__name__, "error": str(error),
        }
        failure_results = pd.concat(
            [failure_results, pd.DataFrame([failure_row])], ignore_index=True
        )
        failure_results.to_csv(failures_path, index=False)
        print(f"  FALHA: {type(error).__name__}: {error}")
        if STOP_ON_ERROR:
            raise

completed_count = (
    metrics_results.loc[metrics_results["strategy"].eq("ENSEMBLE"), "trajectory_index"].nunique()
    if not metrics_results.empty else 0
)
print(f"Trajetórias completas: {completed_count}")
print(f"Falhas registradas: {len(failure_results)}")


[1/5] trajetória 12: executando...


  concluída em 0.1 min
[2/5] trajetória 85: executando...


  concluída em 0.1 min
[3/5] trajetória 174: executando...


  concluída em 0.1 min
[4/5] trajetória 306: executando...


  concluída em 0.1 min
[5/5] trajetória 406: executando...


  concluída em 0.1 min
Trajetórias completas: 5
Falhas registradas: 0


## 5. Resultados descritivos

A tabela resume desempenho e custo. `ALL_PAIRS` e `RANDOM_DENSITY` são controles: o primeiro prevê todos os pares; o segundo conhece apenas a densidade verdadeira, não quais pares são verdadeiros.

In [5]:
if metrics_results.empty:
    if not failure_results.empty:
        display(failure_results)
        raise RuntimeError(
            "Nenhum resultado disponível. As causas reais estão na tabela acima. "
            "Reexecute as seções 1 a 4; RETRY_FAILURES=True refará as trajetórias."
        )
    raise RuntimeError("Nenhum resultado disponível. Execute primeiro a seção 4.")

summary_columns = [
    "average_precision", "roc_auc", "precision", "recall",
    "f1_score", "structural_hamming_distance", "runtime_seconds",
]
descriptive = metrics_results.groupby("strategy")[summary_columns].agg(["mean", "std", "median"])
display(descriptive.round(3))

combination_frequency = (
    selection_results["best_combination"].value_counts().rename_axis("combination")
    .reset_index(name="trajectories")
) if not selection_results.empty else pd.DataFrame()
display(combination_frequency)

if not selection_results.empty and "adaptive_weights" in selection_results:
    adaptive_weight_history = pd.json_normalize(
        selection_results["adaptive_weights"].map(json.loads)
    )
    adaptive_weight_history.index = selection_results["trajectory_index"].astype(int)
    adaptive_weight_history.index.name = "trajectory_index"
    print("Pesos adaptativos por trajetória:")
    display(adaptive_weight_history.round(3))

if not selection_results.empty and "method_redundancy" in selection_results:
        history = pd.json_normalize(
            selection_results["method_redundancy"].map(json.loads)
        )
        history.index = selection_results["trajectory_index"].astype(int)
        history.index.name = "trajectory_index"
        print("Redundância estrutural por método:")
        display(history.round(3))

if not selection_results.empty and "dominant_method_counts" in selection_results:
    dominant_method_history = pd.json_normalize(
        selection_results["dominant_method_counts"].map(json.loads)
    ).fillna(0).astype(int)
    dominant_method_history.index = selection_results["trajectory_index"].astype(int)
    dominant_method_history.index.name = "trajectory_index"
    print("Quantidade de arestas em que cada método foi o especialista local:")
    display(dominant_method_history)

figure = px.box(
    metrics_results, x="strategy", y="average_precision", points="all",
    title="Average Precision por estratégia e trajetória",
)
figure.update_xaxes(tickangle=45)
figure.show()


average_precision               roc_auc                \
                               mean    std median    mean    std median   
strategy                                                                  
ALL_PAIRS                     0.178  0.000  0.178   0.500  0.000  0.500   
DYNOTEARS                     0.245  0.056  0.267   0.471  0.082  0.439   
ENSEMBLE                      0.337  0.062  0.335   0.582  0.050  0.564   
ENSEMBLE_SEM_GATE             0.225  0.048  0.209   0.493  0.056  0.500   
NeuralGrangercMLP             0.250  0.074  0.221   0.548  0.068  0.535   
PCMCI                         0.270  0.152  0.215   0.512  0.095  0.544   
RANDOM_DENSITY                0.320  0.103  0.359   0.510  0.079  0.514   
VARLiNGAM                     0.212  0.121  0.195   0.378  0.225  0.358   

                  precision               recall  ...        f1_score         \
                       mean    std median   mean  ... median     mean    std   
strategy                                          ...                          
ALL_PAIRS             0.178  0.000  0.178  1.000  ...  1.000    0.302  0.000   
DYNOTEARS             0.179  0.010  0.182  0.975  ...  1.000    0.302  0.017   
ENSEMBLE              0.223  0.042  0.217  0.600  ...  0.625    0.325  0.059   
ENSEMBLE_SEM_GATE     0.171  0.026  0.190  0.450  ...  0.500    0.248  0.038   
NeuralGrangercMLP     0.199  0.046  0.185  0.700  ...  0.625    0.309  0.072   
PCMCI                 0.181  0.048  0.192  0.475  ...  0.500    0.262  0.068   
RANDOM_DENSITY        0.200  0.143  0.250  0.200  ...  0.250    0.200  0.143   
VARLiNGAM             0.184  0.006  0.182  1.000  ...  1.000    0.311  0.008   

                         structural_hamming_distance                \
                  median                        mean    std median   
strategy                                                             
ALL_PAIRS          0.302                        37.0  0.000   37.0   
DYNOTEARS          0.308                        36.0  1.225   36.0   
ENSEMBLE           0.323                        20.0  2.000   21.0   
ENSEMBLE_SEM_GATE  0.276                        21.8  1.095   21.0   
NeuralGrangercMLP  0.286                        25.0  2.915   25.0   
PCMCI              0.294                        21.4  2.793   22.0   
RANDOM_DENSITY     0.250                        12.8  2.280   12.0   
VARLiNGAM          0.308                        35.4  1.342   36.0   

                  runtime_seconds                 
                             mean    std  median  
strategy                                          
ALL_PAIRS                   0.000  0.000   0.000  
DYNOTEARS                     NaN    NaN     NaN  
ENSEMBLE                   48.808  8.387  45.621  
ENSEMBLE_SEM_GATE          48.808  8.387  45.621  
NeuralGrangercMLP             NaN    NaN     NaN  
PCMCI                         NaN    NaN     NaN  
RANDOM_DENSITY              0.000  0.000   0.000  
VARLiNGAM                     NaN    NaN     NaN  

[8 rows x 21 columns]

,combination,trajectories
0,PCMCI + DYNOTEARS + NeuralGrangercMLP + VARLiNGAM,5


Pesos adaptativos por trajetória:


,DYNOTEARS,NeuralGrangercMLP,PCMCI,VARLiNGAM
trajectory_index,,,,
12,0.981,1.495,0.565,0.959
85,1.028,1.583,0.516,0.873
174,1.022,1.461,0.698,0.819
306,1.020,1.553,0.575,0.852
406,1.222,1.536,0.395,0.846


Redundância estrutural por método:


,DYNOTEARS,NeuralGrangercMLP,PCMCI,VARLiNGAM
trajectory_index,,,,
12,0.167,0.217,0.185,0.177
85,0.154,0.187,0.136,0.170
174,0.175,0.213,0.185,0.169
306,0.174,0.212,0.164,0.186
406,0.150,0.196,0.161,0.182


Quantidade de arestas em que cada método foi o especialista local:


,DYNOTEARS,NeuralGrangercMLP,PCMCI,VARLiNGAM
trajectory_index,,,,
12,110,33,3,53
85,122,24,4,24
174,105,27,2,55
306,114,30,6,52
406,160,12,0,14


## 6. Análise pareada exploratória

O ensemble é comparado ao PCMCI e aos três componentes congelados. A correção de Holm é calculada, mas cinco trajetórias não fornecem resolução suficiente para significância bilateral de 5%. O resultado principal é a comparação Ensemble versus PCMCI nas trajetórias de validação.

In [6]:
def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    order = np.argsort(values)
    adjusted_sorted = np.maximum.accumulate(
        np.array([(len(values) - rank) * values[index] for rank, index in enumerate(order)])
    )
    adjusted = np.empty_like(values)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted


individual_methods = [*methods, *comparator_methods]
primary_rows = [
    compute_paired_superiority_statistics(
        metrics_results, candidate="ENSEMBLE", baseline=baseline,
        metric="average_precision", higher_is_better=True,
        n_bootstrap=STATISTICAL_BOOTSTRAPS, random_state=RANDOM_STATE,
    )
    for baseline in individual_methods
]
primary_comparison = pd.DataFrame(primary_rows)
primary_comparison["holm_p_value"] = holm_adjust(primary_comparison["wilcoxon_p_value"])
primary_comparison["confirmatory_sample_available"] = (
    primary_comparison["paired_trajectories"] >= MIN_CONFIRMATORY_TRAJECTORIES
)
primary_comparison["superiority_criterion_met"] = (
    primary_comparison["confirmatory_sample_available"]
) & (
    primary_comparison["confidence_interval_low"] > MINIMUM_AP_GAIN
) & (
    primary_comparison["holm_p_value"] < SIGNIFICANCE_LEVEL
) & (
    primary_comparison["win_rate"] >= MINIMUM_WIN_RATE
)
display(primary_comparison.sort_values("baseline").round(4))

secondary_rows = []
for metric, higher_is_better in [("f1_score", True), ("structural_hamming_distance", False)]:
    for baseline in individual_methods:
        secondary_rows.append(compute_paired_superiority_statistics(
            metrics_results, candidate="ENSEMBLE", baseline=baseline,
            metric=metric, higher_is_better=higher_is_better,
            n_bootstrap=STATISTICAL_BOOTSTRAPS, random_state=RANDOM_STATE,
        ))
secondary_comparison = pd.DataFrame(secondary_rows)
display(secondary_comparison.round(4))


,candidate,baseline,metric,paired_trajectories,candidate_mean,baseline_mean,mean_improvement,median_improvement,confidence_interval_low,confidence_interval_high,win_rate,tie_rate,standardized_effect,wilcoxon_p_value,holm_p_value,confirmatory_sample_available,superiority_criterion_met
1,ENSEMBLE,DYNOTEARS,average_precision,5,0.3373,0.2446,0.0927,0.0807,0.0222,0.1575,0.8,0.0,1.0429,0.1250,0.2500,False,False
2,ENSEMBLE,NeuralGrangercMLP,average_precision,5,0.3373,0.2496,0.0877,0.1014,0.0637,0.1118,1.0,0.0,2.8188,0.0625,0.2500,False,False
0,ENSEMBLE,PCMCI,average_precision,5,0.3373,0.2705,0.0669,0.1063,-0.0396,0.1538,0.8,0.0,0.5438,0.3125,0.3125,False,False
3,ENSEMBLE,VARLiNGAM,average_precision,5,0.3373,0.2120,0.1253,0.1519,0.0551,0.1911,1.0,0.0,1.4468,0.0625,0.2500,False,False


,candidate,baseline,metric,paired_trajectories,candidate_mean,baseline_mean,mean_improvement,median_improvement,confidence_interval_low,confidence_interval_high,win_rate,tie_rate,standardized_effect,wilcoxon_p_value
0,ENSEMBLE,PCMCI,f1_score,5,0.3247,0.2615,0.0632,0.0782,-0.0081,0.1374,0.6,0.0,0.6857,0.3125
1,ENSEMBLE,DYNOTEARS,f1_score,5,0.3247,0.3024,0.0224,0.0207,-0.0214,0.0714,0.8,0.0,0.3868,0.4375
2,ENSEMBLE,NeuralGrangercMLP,f1_score,5,0.3247,0.3092,0.0156,-0.0099,-0.0424,0.0981,0.4,0.0,0.1642,1.0000
3,ENSEMBLE,VARLiNGAM,f1_score,5,0.3247,0.3115,0.0133,0.0149,-0.0238,0.0549,0.6,0.0,0.2534,1.0000
4,ENSEMBLE,PCMCI,structural_hamming_distance,5,20.0000,21.4000,1.4000,2.0000,-0.8000,2.8000,0.8,0.0,0.5578,0.4375
5,ENSEMBLE,DYNOTEARS,structural_hamming_distance,5,20.0000,36.0000,16.0000,16.0000,13.8000,17.8000,1.0,0.0,6.2757,0.0625
6,ENSEMBLE,NeuralGrangercMLP,structural_hamming_distance,5,20.0000,25.0000,5.0000,4.0000,2.8000,7.2000,1.0,0.0,1.7150,0.0625
7,ENSEMBLE,VARLiNGAM,structural_hamming_distance,5,20.0000,35.4000,15.4000,15.0000,15.0000,16.2000,1.0,0.0,17.2177,0.0625


## 7. Conclusão automática do estudo piloto

A conclusão abaixo bloqueia afirmações confirmatórias quando há menos de dez trajetórias. Com cinco trajetórias, a redação permitida descreve apenas tendência, magnitude, consistência e custo observados.

In [7]:
passed = primary_comparison.loc[primary_comparison["superiority_criterion_met"], "baseline"].tolist()
failed = primary_comparison.loc[~primary_comparison["superiority_criterion_met"], "baseline"].tolist()
confirmatory_available = bool(primary_comparison["confirmatory_sample_available"].all())

if not confirmatory_available:
    print(
        f"Estudo piloto: somente {int(primary_comparison['paired_trajectories'].min())} "
        f"trajetórias pareadas; o mínimo operacional definido é "
        f"{MIN_CONFIRMATORY_TRAJECTORIES}."
    )
    print(
        "Conclusão permitida: os resultados sugerem ou não sugerem vantagem do ensemble "
        "neste piloto. Reporte ganho médio, intervalo, taxa de vitórias e custo, sem "
        "afirmar superioridade estatisticamente demonstrada."
    )
elif len(passed) == len(individual_methods):
    print(
        "Conclusão permitida: no conjunto de trajetórias avaliado do CausalTime Traffic, "
        "o ensemble apresentou AP superior a todos os algoritmos avulsos registrados, "
        "segundo o critério pré-especificado."
    )
else:
    print(
        "Conclusão permitida: não foi demonstrada superioridade do ensemble sobre todos "
        "os algoritmos avulsos segundo o critério pré-especificado. Reporte quais "
        "comparações foram positivas e quais permaneceram inconclusivas."
    )
print("Comparações sem critério confirmatório atendido:", failed)
print(
    "Não concluir superioridade universal sem repetir o protocolo em datasets com grafos diferentes."
)


Estudo piloto: somente 5 trajetórias pareadas; o mínimo operacional definido é 10.
Conclusão permitida: os resultados sugerem ou não sugerem vantagem do ensemble neste piloto. Reporte ganho médio, intervalo, taxa de vitórias e custo, sem afirmar superioridade estatisticamente demonstrada.
Comparações sem critério confirmatório atendido: ['PCMCI', 'DYNOTEARS', 'NeuralGrangercMLP', 'VARLiNGAM']
Não concluir superioridade universal sem repetir o protocolo em datasets com grafos diferentes.
